In [21]:
import cv2
import mediapipe as mp
import warnings
import numpy as np
import math
warnings.filterwarnings("ignore")

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_hands = mp.solutions.hands

cap = cv2.VideoCapture(0)

drawing_mode = False
canvas = None
prev = None

with mp_hands.Hands(model_complexity=0, min_detection_confidence=0.5, min_tracking_confidence=0.5) as hands:

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            break

        h, w, _ = image.shape
        image = cv2.resize(image, (w // 2, h // 2))
        h, w, _ = image.shape

        if canvas is None:
            canvas = np.zeros_like(image)

        image.flags.writeable = False
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb_image)
        image.flags.writeable = True

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                landmarks = hand_landmarks.landmark
                
                #제스쳐 판별용 로직
                fingers = []
                
                #엄지
                fingers.append(1 if landmarks[4].x > landmarks[3].x else 0)
                
                tips = [8, 12, 16, 20]
                pips = [6, 10, 14, 18]
                
                #나머지 4손가락
                for tip, pip in zip(tips, pips):
                    fingers.append(1 if landmarks[tip].y < landmarks[pip].y else 0)

                #엄지와 검지 
                thumbs_tip = hand_landmarks.landmark[4]
                index_tip = hand_landmarks.landmark[8] 
                index_pip = hand_landmarks.landmark[7]

                #엄지와 검지 픽셀좌표
                thumbs_tip_pxpy = (int(thumbs_tip.x * w), int(thumbs_tip.y * h)) 
                index_tip_pxpy = (int(index_tip.x * w), int(index_tip.y * h))
                
                #엄지 검지 사이 거리 
                dis_thumb2in = math.hypot(thumbs_tip.x - index_tip.x , thumbs_tip.y - index_tip.y) 
                cv2.line(image, thumbs_tip_pxpy, index_tip_pxpy, (255,0,0), 5)
                cv.putText(image, f'Distance : int{dis_thumb2in}', (50,50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

                # 손목 
                wrist = hand_landmarks.landmark[0]
                wrist_pxpy = (int(wrist.x * w), int(wrist.y * h))
                
                # 손목과 검지 중간 사이 거리 
                dis_wri2in = math.hypot(wrist.x - index_pip.x, wrist.y - index_pip.y)

                now = index_tip_pxpy
                
                if fingers == [0,0,0,0,1]:
                    cv2.putText(image, 'FontSize', (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                
                if not drawing_mode:
                    if fingers == [0,1,0,0,0] and dis_wri2in > 250 :
                        drawing_mode = True
                        prev = now
                
                else: 
                    if not fingers == [0,0,0,0,0]:
                        drawing_mode = False
                        prev = None
                    else:
                        if prev is not None:
                            cv2.line(canvas, prev, now, (255, 200, 100), 5)
                            prev = now

                
                # mp_drawing.draw_landmarks(
                #     image,
                #     hand_landmarks,
                #     mp_hands.HAND_CONNECTIONS,
                #     mp_drawing_styles.get_default_hand_landmarks_style(),
                #     mp_drawing_styles.get_default_hand_connections_style()
                # )
        else:
            prev = None

        result = cv2.add(image, canvas)

        cv2.imshow('Hands', result)
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

NameError: name 'cv' is not defined

In [1]:
import cv2
import mediapipe as mp
import warnings
import numpy as np
import math

warnings.filterwarnings("ignore")

mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

cap = cv2.VideoCapture(0)

drawing_mode = False
canvas = None
prev = None

with mp_hands.Hands(
    model_complexity=0,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as hands:

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            break

        h, w, _ = image.shape
        image = cv2.resize(image, (w // 2, h // 2))
        h, w, _ = image.shape

        if canvas is None or canvas.shape != image.shape:
            canvas = np.zeros_like(image)

        image.flags.writeable = False
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb_image)
        image.flags.writeable = True

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(
                    image,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS
                )

                landmarks = hand_landmarks.landmark

                # 주요 포인트
                thumb_tip = landmarks[4]
                index_tip = landmarks[8]
                index_pip = landmarks[6]
                pinky_tip = landmarks[20]
                pinky_pip = landmarks[18]
                wrist = landmarks[0]

                # 픽셀 좌표
                thumb_pt = (int(thumb_tip.x * w), int(thumb_tip.y * h))
                index_pt = (int(index_tip.x * w), int(index_tip.y * h))
                index_pip_pt = (int(index_pip.x * w), int(index_pip.y * h))
                wrist_pt = (int(wrist.x * w), int(wrist.y * h))

                now = index_pt

                # 거리
                dis_thumb2in = math.hypot(
                    thumb_pt[0] - index_pt[0],
                    thumb_pt[1] - index_pt[1]
                )

                dis_wri2in = math.hypot(
                    wrist_pt[0] - index_pip_pt[0],
                    wrist_pt[1] - index_pip_pt[1]
                )

                # 손가락 상태
                index_open = index_tip.y < index_pip.y
                pinky_open = pinky_tip.y < pinky_pip.y

                cv2.putText(image, f'Dis(thumb-index): {dis_thumb2in:.1f}', (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
                cv2.putText(image, f'Dis(wrist-index): {dis_wri2in:.1f}', (30, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
                cv2.line(image, thumb_pt, index_pt, (255, 0, 0), 3)
                
                # 새끼 손가락 펼치면 글자 표시
                if pinky_open:
                    cv2.putText(image, 'FontSize', (30, 160), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

                # 시작: 아직 안 그리고 있을 때
                if not drawing_mode:
                    if index_open and dis_wri2in > 250:
                        drawing_mode = True
                        prev = now

                # 유지/종료
                else:
                    if not index_open:
                        drawing_mode = False
                        prev = None
                    else:
                        if prev is not None:
                            cv2.line(canvas, prev, now, (255, 200, 100), 5)
                        prev = now

                # 손 하나만 쓸 거면 여기서 끊어도 됨
                break

        else:
            drawing_mode = False
            prev = None

        result = cv2.add(image, canvas)

        cv2.imshow('Hands', result)
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [20]:
import cv2
import mediapipe as mp
import numpy as np
import math
import warnings

warnings.filterwarnings("ignore")

mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands

cap = cv2.VideoCapture(0)

canvas = None
prev = None

tool = None
draw_enabled = False
draw_color = (0, 255, 255)
thickness = 5

SIDEBAR_W = 180

buttons = [
    {"name": "pen",     "label": "PEN",     "x1": 10, "y1": 20,  "x2": 170, "y2": 70},
    {"name": "eraser",  "label": "ERASER",  "x1": 10, "y1": 90,  "x2": 170, "y2": 140},
    {"name": "red",     "label": "RED",     "x1": 10, "y1": 160, "x2": 170, "y2": 210},
    {"name": "green",   "label": "GREEN",   "x1": 10, "y1": 230, "x2": 170, "y2": 280},
    {"name": "blue",    "label": "BLUE",    "x1": 10, "y1": 300, "x2": 170, "y2": 350},
    {"name": "thin",    "label": "THIN",    "x1": 10, "y1": 370, "x2": 170, "y2": 420},
    {"name": "thick",   "label": "THICK",   "x1": 10, "y1": 440, "x2": 170, "y2": 490},
    {"name": "clear",   "label": "CLEAR",   "x1": 10, "y1": 510, "x2": 170, "y2": 560},
]

click_lock = False

def draw_sidebar(img, hovered_button, tool, draw_color, thickness):
    overlay = img.copy()

    cv2.rectangle(overlay, (0, 0), (SIDEBAR_W, img.shape[0]), (40, 40, 40), -1)
    cv2.addWeighted(overlay, 0.9, img, 0.1, 0, img)

    for btn in buttons:
        x1, y1, x2, y2 = btn["x1"], btn["y1"], btn["x2"], btn["y2"]

        fill_color = (80, 80, 80)
        border_color = (180, 180, 180)

        if hovered_button == btn["name"]:
            fill_color = (120, 120, 120)

        if btn["name"] == tool:
            border_color = (0, 255, 255)

        if btn["name"] == "red":
            fill_color = (0, 0, 180)
        elif btn["name"] == "green":
            fill_color = (0, 180, 0)
        elif btn["name"] == "blue":
            fill_color = (180, 0, 0)

        cv2.rectangle(img, (x1, y1), (x2, y2), fill_color, -1)
        cv2.rectangle(img, (x1, y1), (x2, y2), border_color, 2)

        cv2.putText(
            img, btn["label"], (x1 + 15, y1 + 33),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2
        )

    tool_text = tool.upper() if tool is not None else "NONE"
    cv2.putText(img, f"TOOL : {tool_text}", (15, img.shape[0] - 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.putText(img, f"SIZE : {thickness}", (15, img.shape[0] - 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.putText(img, "COLOR", (15, img.shape[0] - 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, draw_color, 2)


def get_hovered_button(x, y):
    for btn in buttons:
        if btn["x1"] <= x <= btn["x2"] and btn["y1"] <= y <= btn["y2"]:
            return btn["name"]
    return None


with mp_hands.Hands(
    model_complexity=0,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as hands:

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            break
        image = cv2.flip(image, 1)

        h, w, _ = image.shape
        image = cv2.resize(image, (960, 720))
        h, w, _ = image.shape

        if canvas is None or canvas.shape != image.shape:
            canvas = np.zeros_like(image)

        image.flags.writeable = False
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb_image)
        image.flags.writeable = True

        hovered_button = None
        pinch_clicked = False
        index_open = False
        index_pt = None
        pinch_clicked2 = False

        if results.multi_hand_landmarks:
            hand_landmarks = results.multi_hand_landmarks[0]
            mp_drawing.draw_landmarks(image, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            landmarks = hand_landmarks.landmark

            thumb_tip = landmarks[4]
            index_tip = landmarks[8]
            index_pip = landmarks[6]
            middle_tip = landmarks[12]
            middle_pip = landmarks[10]
            

            thumb_pt = (int(thumb_tip.x * w), int(thumb_tip.y * h))
            index_pt = (int(index_tip.x * w), int(index_tip.y * h))
            index_pip_pt = (int(index_pip.x * w), int(index_pip.y * h))
            middle_pt = (int(middle_tip.x * w), int(middle_tip.y * h))
            middle_pip_pt = (int(middle_pip.x * w), int(middle_pip.y * h))

            pinch_distance = math.hypot(
                thumb_pt[0] - index_pt[0],
                thumb_pt[1] - index_pt[1]
            )
            pinch_distance2 = math.hypot(
                index_pt[0] - middle_pt[0],
                index_pt[1] - middle_pt[1]
            )

            index_open = index_tip.y < index_pip.y

            hovered_button = get_hovered_button(index_pt[0], index_pt[1])

            cv2.putText(
                image,
                f"PINCH: {pinch_distance:.1f}",
                (200, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 255),
                2
            )

            if pinch_distance < 45:
                pinch_clicked = True

            # 1) 사이드바 버튼 선택: hover + pinch 일 때만
            if index_pt[0] <= SIDEBAR_W:
                if pinch_clicked and not click_lock and hovered_button is not None:
                    click_lock = True
            
                    if hovered_button == "pen":
                        tool = "pen"
                        draw_enabled = True
            
                    elif hovered_button == "eraser":
                        tool = "eraser"
                        draw_enabled = True
            
                    elif hovered_button == "red":
                        draw_color = (0, 0, 255)
            
                    elif hovered_button == "green":
                        draw_color = (0, 255, 0)
            
                    elif hovered_button == "blue":
                        draw_color = (255, 0, 0)
            
                    elif hovered_button == "thin":
                        thickness = 3
            
                    elif hovered_button == "thick":
                        thickness = 12
            
                    elif hovered_button == "clear":
                        canvas = np.zeros_like(image)


                # 사이드바 안에서는 절대 그리지 않음
                prev = None

            # 2) 캔버스 영역: pen/eraser 모드에서 pinch 유지할 때만 그리기
            else:
                if draw_enabled and index_open and tool in ["pen", "eraser"]:
                    if prev is not None:
                        if tool == "pen":
                            cv2.line(canvas, prev, index_pt, draw_color, thickness)
                        elif tool == "eraser":
                            cv2.line(canvas, prev, index_pt, (0, 0, 0), 40)
                    prev = index_pt
                else:
                    prev = None

            # 핀치 떼면 다시 클릭 가능
            if not pinch_clicked:
                click_lock = False

        else:
            prev = None
            click_lock = False

        result = cv2.add(image, canvas)
        draw_sidebar(result, hovered_button, tool, draw_color, thickness)

        cv2.imshow('Air Whiteboard', result)
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [18]:
import cv2
import mediapipe as mp
import numpy as np
import math
import warnings

warnings.filterwarnings("ignore")

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

# =========================
# 기본 설정
# =========================
CAM_INDEX = 0

WIN_W = 1280
WIN_H = 720

SIDEBAR_W = 170
PREVIEW_W = 220

BOARD_X1 = SIDEBAR_W
BOARD_X2 = WIN_W - PREVIEW_W
BOARD_W = BOARD_X2 - BOARD_X1
BOARD_H = WIN_H

PINCH_THRESHOLD = 42      # 엄지 + 검지
CLEAR_THRESHOLD = 30      # 검지 + 중지
ERASER_SIZE = 46

SMOOTHING_ALPHA = 0.32
INTERPOLATION_STEPS = 8

# BGR
WHITE = (255, 255, 255)
RED = (0, 0, 255)
YELLOW = (0, 255, 255)
BLUE = (255, 0, 0)

# =========================
# 상태값
# =========================
tool = "pen"
draw_enabled = True
draw_color = WHITE
thickness = 5

board_canvas = None
prev_draw_pt = None
smooth_pt = None

click_lock = False
clear_lock = False

# =========================
# 버튼 정의
# =========================
buttons = [
    {"name": "pen",    "kind": "tool",  "x1": 18, "y1": 30,  "x2": 152, "y2": 82},
    {"name": "eraser", "kind": "tool",  "x1": 18, "y1": 96,  "x2": 152, "y2": 148},

    {"name": "white",  "kind": "color", "x1": 18, "y1": 198, "x2": 152, "y2": 250},
    {"name": "red",    "kind": "color", "x1": 18, "y1": 264, "x2": 152, "y2": 316},
    {"name": "yellow", "kind": "color", "x1": 18, "y1": 330, "x2": 152, "y2": 382},
    {"name": "blue",   "kind": "color", "x1": 18, "y1": 396, "x2": 152, "y2": 448},

    {"name": "thin",   "kind": "size",  "x1": 18, "y1": 506, "x2": 152, "y2": 558},
    {"name": "thick",  "kind": "size",  "x1": 18, "y1": 572, "x2": 152, "y2": 624},

    {"name": "clear",  "kind": "tool",  "x1": 18, "y1": 652, "x2": 152, "y2": 704},
]

# =========================
# 유틸
# =========================
def get_hovered_button(x, y):
    for btn in buttons:
        if btn["x1"] <= x <= btn["x2"] and btn["y1"] <= y <= btn["y2"]:
            return btn["name"]
    return None

def smooth_point(prev_smooth, current_pt, alpha=0.32):
    if prev_smooth is None:
        return current_pt
    sx = int(prev_smooth[0] * (1 - alpha) + current_pt[0] * alpha)
    sy = int(prev_smooth[1] * (1 - alpha) + current_pt[1] * alpha)
    return (sx, sy)

def color_name_from_bgr(color):
    if color == WHITE:
        return "WHITE"
    if color == RED:
        return "RED"
    if color == YELLOW:
        return "YELLOW"
    if color == BLUE:
        return "BLUE"
    return "CUSTOM"

def is_selected(btn_name):
    global tool, draw_color, thickness

    if btn_name == tool:
        return True
    if btn_name == "white" and draw_color == WHITE:
        return True
    if btn_name == "red" and draw_color == RED:
        return True
    if btn_name == "yellow" and draw_color == YELLOW:
        return True
    if btn_name == "blue" and draw_color == BLUE:
        return True
    if btn_name == "thin" and thickness == 3:
        return True
    if btn_name == "thick" and thickness == 10:
        return True
    return False

# =========================
# UI
# =========================
def draw_sidebar(frame, hovered_button):
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (SIDEBAR_W, WIN_H), (24, 24, 26), -1)
    cv2.addWeighted(overlay, 0.94, frame, 0.06, 0, frame)

    cv2.putText(frame, "TOOLS", (20, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (170, 170, 170), 1)
    cv2.putText(frame, "COLOR", (20, 188), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (170, 170, 170), 1)
    cv2.putText(frame, "SIZE",  (20, 496), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (170, 170, 170), 1)

    for btn in buttons:
        x1, y1, x2, y2 = btn["x1"], btn["y1"], btn["x2"], btn["y2"]
        name = btn["name"]
        kind = btn["kind"]

        fill = (44, 44, 48)
        border = (78, 78, 84)
        text_color = (240, 240, 240)

        if hovered_button == name:
            fill = (58, 58, 64)

        if is_selected(name):
            border = (245, 245, 245)

        if name == "clear":
            fill = (58, 46, 50)

        # 완전 일반 사각형 카드
        cv2.rectangle(frame, (x1, y1), (x2, y2), fill, -1)
        cv2.rectangle(frame, (x1, y1), (x2, y2), border, 2)

        if kind == "tool":
            label = name.upper()
            font_scale = 0.62 if name != "eraser" else 0.55
            text_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, 2)[0]
            tx = x1 + (x2 - x1 - text_size[0]) // 2
            ty = y1 + (y2 - y1 + text_size[1]) // 2
            cv2.putText(frame, label, (tx, ty), cv2.FONT_HERSHEY_SIMPLEX, font_scale, text_color, 2, cv2.LINE_AA)

        elif kind == "color":
            color_map = {
                "white": WHITE,
                "red": RED,
                "yellow": YELLOW,
                "blue": BLUE,
            }
            c = color_map[name]
            center = ((x1 + x2) // 2, (y1 + y2) // 2)

            fill_color = (245, 245, 245) if c == WHITE else c
            outline = (180, 180, 180) if c == WHITE else (255, 255, 255)

            cv2.circle(frame, center, 15, fill_color, -1, cv2.LINE_AA)
            cv2.circle(frame, center, 15, outline, 2, cv2.LINE_AA)

        elif kind == "size":
            cy = (y1 + y2) // 2
            if name == "thin":
                cv2.line(frame, (x1 + 24, cy), (x2 - 24, cy), (245, 245, 245), 2, cv2.LINE_AA)
            else:
                cv2.line(frame, (x1 + 24, cy), (x2 - 24, cy), (245, 245, 245), 8, cv2.LINE_AA)

    # 상태 패널도 그냥 사각형
    panel_y1 = WIN_H - 84
    panel_y2 = WIN_H - 16

    cv2.rectangle(frame, (14, panel_y1), (SIDEBAR_W - 14, panel_y2), (34, 34, 38), -1)
    cv2.rectangle(frame, (14, panel_y1), (SIDEBAR_W - 14, panel_y2), (74, 74, 80), 1)

    tool_text = tool.upper() if tool else "NONE"
    color_text = color_name_from_bgr(draw_color)

    cv2.putText(frame, tool_text, (24, panel_y1 + 22), cv2.FONT_HERSHEY_SIMPLEX, 0.46, (240, 240, 240), 1, cv2.LINE_AA)
    cv2.putText(frame, color_text, (24, panel_y1 + 46), cv2.FONT_HERSHEY_SIMPLEX, 0.40, (195, 195, 195), 1, cv2.LINE_AA)
    cv2.putText(frame, f"{thickness}px", (100, panel_y1 + 46), cv2.FONT_HERSHEY_SIMPLEX, 0.40, (195, 195, 195), 1, cv2.LINE_AA)

    preview_color = (245, 245, 245) if draw_color == WHITE else draw_color
    outline = (180, 180, 180) if draw_color == WHITE else (255, 255, 255)
    cv2.circle(frame, (SIDEBAR_W - 28, panel_y1 + 24), 10, preview_color, -1, cv2.LINE_AA)
    cv2.circle(frame, (SIDEBAR_W - 28, panel_y1 + 24), 10, outline, 2, cv2.LINE_AA)

def draw_board_background(frame):
    cv2.rectangle(frame, (BOARD_X1, 0), (BOARD_X2, WIN_H), (38, 92, 40), -1)
    cv2.rectangle(frame, (BOARD_X1, 0), (BOARD_X2, WIN_H), (108, 70, 30), 16)
    cv2.rectangle(frame, (BOARD_X1 + 8, 8), (BOARD_X2 - 8, WIN_H - 8), (86, 58, 28), 2)

def draw_preview_area(frame, cam_full):
    x1 = BOARD_X2
    x2 = WIN_W

    preview = cv2.resize(cam_full[:, x1:x2], (PREVIEW_W, WIN_H))
    frame[:, x1:x2] = preview

    dark = np.zeros_like(frame[:, x1:x2])
    frame[:, x1:x2] = cv2.addWeighted(frame[:, x1:x2], 0.82, dark, 0.18, 0)

    cv2.rectangle(frame, (x1, 0), (x2, WIN_H), (38, 38, 44), 2)

# =========================
# 메인
# =========================
cap = cv2.VideoCapture(CAM_INDEX)

with mp_hands.Hands(
    model_complexity=0,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as hands:

    while cap.isOpened():
        success, image = cap.read()
        if not success:
            break

        image = cv2.flip(image, 1)
        image = cv2.resize(image, (WIN_W, WIN_H))

        h, w, _ = image.shape

        if board_canvas is None or board_canvas.shape[:2] != (BOARD_H, BOARD_W):
            board_canvas = np.zeros((BOARD_H, BOARD_W, 3), dtype=np.uint8)

        hovered_button = None
        pinch_clicked = False
        pinch_clicked2 = False

        image.flags.writeable = False
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb_image)
        image.flags.writeable = True

        if results.multi_hand_landmarks:
            hand_landmarks = results.multi_hand_landmarks[0]
            landmarks = hand_landmarks.landmark

            thumb_tip = landmarks[4]
            index_tip = landmarks[8]
            index_pip = landmarks[6]
            middle_tip = landmarks[12]

            thumb_pt = (int(thumb_tip.x * w), int(thumb_tip.y * h))
            raw_index_pt = (int(index_tip.x * w), int(index_tip.y * h))
            middle_pt = (int(middle_tip.x * w), int(middle_tip.y * h))
            index_pip_pt = (int(index_pip.x * w), int(index_pip.y * h))

            pinch_distance = math.hypot(
                thumb_pt[0] - raw_index_pt[0],
                thumb_pt[1] - raw_index_pt[1]
            )

            clear_distance = math.hypot(
                raw_index_pt[0] - middle_pt[0],
                raw_index_pt[1] - middle_pt[1]
            )

            index_open = index_tip.y < index_pip.y

            smooth_pt = smooth_point(smooth_pt, raw_index_pt, SMOOTHING_ALPHA)
            index_pt = smooth_pt

            hovered_button = get_hovered_button(index_pt[0], index_pt[1])

            if pinch_distance < PINCH_THRESHOLD:
                pinch_clicked = True

            if clear_distance < CLEAR_THRESHOLD:
                pinch_clicked2 = True

            # 검지+중지 핀치 = 전체 지우기
            if pinch_clicked2 and not clear_lock:
                board_canvas = np.zeros_like(board_canvas)
                prev_draw_pt = None
                clear_lock = True

            # 사이드바
            if index_pt[0] < SIDEBAR_W:
                if pinch_clicked and not click_lock and hovered_button is not None:
                    click_lock = True

                    if hovered_button == "pen":
                        tool = "pen"
                        draw_enabled = True

                    elif hovered_button == "eraser":
                        tool = "eraser"
                        draw_enabled = True

                    elif hovered_button == "white":
                        draw_color = WHITE

                    elif hovered_button == "red":
                        draw_color = RED

                    elif hovered_button == "yellow":
                        draw_color = YELLOW

                    elif hovered_button == "blue":
                        draw_color = BLUE

                    elif hovered_button == "thin":
                        thickness = 3

                    elif hovered_button == "thick":
                        thickness = 10

                    elif hovered_button == "clear":
                        board_canvas = np.zeros_like(board_canvas)
                        prev_draw_pt = None

                prev_draw_pt = None

            # 칠판
            elif BOARD_X1 <= index_pt[0] < BOARD_X2:
                board_pt = (index_pt[0] - BOARD_X1, index_pt[1])

                if draw_enabled and index_open and tool in ["pen", "eraser"]:
                    if prev_draw_pt is not None:
                        for i in range(1, INTERPOLATION_STEPS + 1):
                            t = i / INTERPOLATION_STEPS
                            ix = int(prev_draw_pt[0] * (1 - t) + board_pt[0] * t)
                            iy = int(prev_draw_pt[1] * (1 - t) + board_pt[1] * t)

                            if tool == "pen":
                                cv2.circle(board_canvas, (ix, iy), max(1, thickness // 2), draw_color, -1, cv2.LINE_AA)
                            else:
                                cv2.circle(board_canvas, (ix, iy), ERASER_SIZE // 2, (0, 0, 0), -1, cv2.LINE_AA)

                        if tool == "pen":
                            cv2.line(board_canvas, prev_draw_pt, board_pt, draw_color, thickness, cv2.LINE_AA)
                        else:
                            cv2.line(board_canvas, prev_draw_pt, board_pt, (0, 0, 0), ERASER_SIZE, cv2.LINE_AA)

                    prev_draw_pt = board_pt
                else:
                    prev_draw_pt = None

            else:
                prev_draw_pt = None

            if not pinch_clicked:
                click_lock = False

            if not pinch_clicked2:
                clear_lock = False

        else:
            prev_draw_pt = None
            smooth_pt = None
            click_lock = False
            clear_lock = False

        # 화면 합성
        result = np.zeros_like(image)
        result[:] = (20, 20, 22)

        draw_board_background(result)

        board_bg = result[:, BOARD_X1:BOARD_X2]
        result[:, BOARD_X1:BOARD_X2] = cv2.add(board_bg, board_canvas)

        draw_preview_area(result, image)
        draw_sidebar(result, hovered_button)

        if smooth_pt is not None and BOARD_X1 <= smooth_pt[0] < BOARD_X2:
            pointer_color = (245, 245, 245) if tool == "pen" else (200, 200, 200)
            cv2.circle(result, smooth_pt, 7, pointer_color, 2, cv2.LINE_AA)

        if results.multi_hand_landmarks:
            preview_layer = image.copy()
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(
                    preview_layer,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS,
                    mp_drawing.DrawingSpec(color=(190, 190, 190), thickness=2, circle_radius=2),
                    mp_drawing.DrawingSpec(color=(120, 120, 120), thickness=2, circle_radius=2)
                )

            result[:, BOARD_X2:WIN_W] = cv2.addWeighted(
                result[:, BOARD_X2:WIN_W], 0.7,
                preview_layer[:, BOARD_X2:WIN_W], 0.3,
                0
            )

        cv2.imshow("Air Whiteboard", result)

        key = cv2.waitKey(10) & 0xFF
        if key == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()